<div class='heading'>
    <div style='float:left;'><h1>CPSC 8810: Machine Learning for Graphs</h1></div>
     <img style="float: right; padding-right: 10px" width="100" src="https://raw.githubusercontent.com/bsethwalker/clemson-cs4300/main/images/clemson_paw.png"> </div>
     </div>

**Clemson University**<br>
**Instructor(s):** Aaron Masino <br>

## Lab 5: Graph Transformers
This notebook graph transformers. It illustrates the use graph transformers in combination with node and graph level prediction heads for classification tasks. In each case, the prediction model leverages GNN layers that follow from the message passing framework discussed in class. Three datasets are considered to highlight different tasks and modeling considerations: Cora citation network, TU Dortmund University Enzymes data, and a large synthetic network. Finally, the notebook also introduces use of the [PyTorch Lightning](https://lightning.ai/pytorch-lightning) libray which will support data loading and model training.

### Learning Objectives
1. Understand the limitations of applying standard Transformer Encoder Layer from PyTorch to graph data
2. Apply random walks for positional encoding of nodes in graph transformers
2. Apply eignevectors of Graph Laplacian matrix as positional encode onf nodes in graph transformers
3. Create deep learning models on graph data using TransformerConv layers.
5. Apply model parameter tuning and model evaluation to select the best model.
6. Evaluate the performance of transformer models for node and graph classification tasks.

In [ ]:
# Google Colab setup
# mount the google drive - this is necessary to access supporting src
from google.colab import drive
drive.mount("/content/drive")

# Create output directory
import os
from pathlib import Path

# Create output directory
output_dir = Path("/content/drive/MyDrive/Colab Notebooks/ml4g/labs/output/lab_05")
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory created: {output_dir}")

# data directory
data_dir = Path('/content/drive/MyDrive/Colab Notebooks/ml4g/data')
data_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
!pip3 install torch_geometric
!pip install lightning

In [ ]:
# Data manipulation and analysis
import numpy as np
import pandas as pd
from pathlib import Path
import os

# Graph analysis
import networkx as nx

# PyTorch and PyTorch Geometric
import torch
from torch.utils.data import DataLoader
import torch.nn.functional as F
# import torch.nn as tnn so as not to conflict with nn from PyTorch Geometric
from torch import nn as tnn
from torch_geometric.datasets import Planetoid, TUDataset, ZINC
from torch_geometric.utils import to_networkx
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import RandomNodeSplit, AddLaplacianEigenvectorPE, AddRandomWalkPE
from torch_geometric.nn import TransformerConv, global_mean_pool
from torch_geometric.seed import seed_everything

# PyTorch Lightning
import lightning as L
from lightning.pytorch import seed_everything

# Machine Learning
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve, auc)
from sklearn.metrics import roc_curve, auc

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Setup
plt.style.use('default')
sns.set_palette("husl")

# Reusable random state
RANDOM_STATE = 654321
seed_everything(RANDOM_STATE)

# Pandas set max columns to display all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# set device
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

In [ ]:
def plot_roc_curves(y_true, y_pred_proba, class_names, title="ROC Curves"):
    """
    Plot ROC curves for multi-class classification

    Args:
        y_true: True labels (numpy array or tensor)
        y_pred_proba: Predicted probabilities for each class (numpy array or tensor)
        class_names: List of class names
        title: Plot title
    """

    # Convert to numpy if tensors
    if torch.is_tensor(y_true):
        y_true = y_true.cpu().detach().numpy()
    if torch.is_tensor(y_pred_proba):
        y_pred_proba = y_pred_proba.cpu().detach().numpy()

    # Binarize the labels for multi-class ROC
    y_true_bin = label_binarize(y_true, classes=range(len(class_names)))

    plt.figure(figsize=(8,6))
    colors = [f'C{i}' for i in range(len(class_names))]

    for i in range(len(class_names)):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, color=colors[i], lw=2,
                 label=f'{class_names[i]} (AUC = {roc_auc:.3f})')

    plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random (AUC = 0.500)')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_confusion_matrix(y_true, y_pred, class_names, title="Confusion Matrix"):
    """
    Plot confusion matrix with class names

    Args:
        y_true: True labels (numpy array or tensor)
        y_pred: Predicted labels (numpy array or tensor)
        class_names: List of class names
        title: Plot title
    """

    # Convert to numpy if tensors
    if torch.is_tensor(y_true):
        y_true = y_true.cpu().numpy()
    if torch.is_tensor(y_pred):
        y_pred = y_pred.cpu().numpy()

    cm = confusion_matrix(y_true, y_pred, normalize='pred')

    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

---
# Part 1 Naive Graph Transformer Implementation

Let's try implementing a graph transformer using the standard Transformer modules as implemented for sequence data (e.g., text) in PyTorch as described conceptually in class. In this approach, we formulate the Transformer input as a sequence of node features augmented with positional encodings and apply the full attention mechansim of the Transformer, i.e., attention is computed between every pair of nodes, even if they're not connected.


## 1.1 The Enzyme dataset
First, let's load the ENZYMES dataset from the PyTorch Geometric [TUDataset](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.datasets.TUDataset.html#torch_geometric.datasets.TUDataset) module. The dataset consists of 600 graphs. Each node has 3 features, though interestingly there seems to be no online documentation on what they represent. However, we can examine the distinct values of the features to get a sense of the type of variable represented by the feature. It turns out that each feature is binary.

In [ ]:
enz = TUDataset(root=f'{data_dir}', name='ENZYMES')
enz_class_map={0: 'Oxidoreductases', 1: 'Transferases', 2: 'Hydrolases', 3: 'Lyases', 4: 'Isomerases', 5: 'Ligases'}

print("\nDataset Summary:")
enz.print_summary()

Recall, that it will be convenient for us to use the PyTorch Lightning Data Module to graph data. Here, we build on the data module from the previous lab. However, we need to make changes to account for the positional encodings. In this case, we will pass in a PyG Transform (not a Transform*er*) that will process the graph data to add positional encoding information.

In [ ]:
class EnzymePEDataModule(L.LightningDataModule):
    def __init__(self, batch_size=32, test_fraction = 0.2, val_fraction = 0.1, shuffle = True, pe_transform=None):
        super().__init__()
        if pe_transform is None:
            self.dataset = TUDataset(root=f'{data_dir}', name='ENZYMES')
        else:
            self.dataset = TUDataset(root=f'{data_dir}', name='ENZYMES', pre_transform=pe_transform, force_reload=True)
        self.batch_size = batch_size
        self.test_fraction = test_fraction
        self.val_fraction = val_fraction
        self.class_name_map = {0: 'Oxidoreductases', 1: 'Transferases', 2: 'Hydrolases', 3: 'Lyases', 4: 'Isomerases', 5: 'Ligases'}
        self.num_classes = self.dataset.num_classes
        self.num_node_features = self.dataset.num_node_features
        self.shuffle = shuffle
        self._setup()

    def _setup(self, stage=None):
        test_size = int(len(self.dataset) * self.test_fraction)
        val_size = int(len(self.dataset) * self.val_fraction)
        train_size = len(self.dataset) - test_size - val_size

        self.train_dataset, self.val_dataset, self.test_dataset = torch.utils.data.random_split(
            self.dataset, [train_size, val_size, test_size])

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def test_dataloader(self):
        return  DataLoader(self.test_dataset, batch_size=self.batch_size)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size)

Let's construct our datamodule using the [AddRandomWalkPE](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.transforms.AddRandomWalkPE.html#torch_geometric.transforms.AddRandomWalkPE) positional encodding transform from PyG.

For more information see [GRAPH NEURAL NETWORKS WITH LEARNABLE STRUCTURAL AND POSITIONAL REPRESENTATIONS](https://arxiv.org/pdf/2110.07875)

In [ ]:
seed_everything(RANDOM_STATE)
pe_dim = 8
dm = EnzymePEDataModule(batch_size=2, pe_transform=AddRandomWalkPE(walk_length=pe_dim, attr_name='pe'))
print(f"Number of classes: {dm.num_classes}")
loader = dm.train_dataloader()
cnt = 0
max_cnt = 2
for step, data in enumerate(loader):
    print(f'Step {step + 1}:')
    print('=======')
    print(f'Number of graphs in the current batch: {data.num_graphs}')
    print(data)
    print()
    cnt += 1
    if cnt == max_cnt:
        break

## 1.2 Let's build a Naive Transformer
Here we will treat the nodes as a sequence and pass them to a standard transformer encoder as implemented in PyTorch. The input to the transformer layer is an embedding, i.e., a linear project of the concatenated node features and positional encoding. We then "pool" the nodes by passing the transformer embedding outputs through a non-linear activation and then a linear project to the output logits.

In [ ]:
class GraphSequenceTransformer(tnn.Module):
    def __init__(self, node_feature_dim, pe_dim, num_classes, hidden_dim=64, num_heads=2):
        super().__init__()

        # Before we pass to transformer, we want to project node features to hidden dimension
        # Embed node features + positional encoding to transformer dimension
        self.node_embedding = tnn.Linear(node_feature_dim + pe_dim, hidden_dim)

        # Standard PyTorch Transformer layer
        self.transformer_layer = tnn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 2,
            batch_first=True,
            dropout=0.1,
            activation='relu'
        )

        # Graph-level prediction head
        self.graph_pooling = tnn.Linear(hidden_dim, hidden_dim)
        self.predictor = tnn.Linear(hidden_dim, num_classes)

    def forward(self, batch):
        """
        Process each graph individually (no real batching!)
        """
        #************************************************************************
        # Convert PyG batch back to individual graphs - THIS IS A HUGE LIMITATION. Let's discuss why!
        #************************************************************************
        graphs = batch.to_data_list()
        batch_outputs = []

        for graph in graphs:
            # Combine node features with random walk positional encoding
            node_features = torch.cat([graph.x, graph.pe], dim=-1)  # [num_nodes, node_dim + pe_dim]

            # Embed to transformer dimension
            x = self.node_embedding(node_features)  # [num_nodes, hidden_dim]

            # Add batch dimension for transformer (treating nodes as sequence)
            x = x.unsqueeze(0)  # [1, num_nodes, hidden_dim]

            # Apply transformer - each node attends to ALL other nodes
            # (This ignores the actual graph structure!)
            x = self.transformer_layer(x)  # [1, num_nodes, hidden_dim]

            # Global mean pooling for graph-level prediction
            graph_repr = x.squeeze(0).mean(dim=0)  # [hidden_dim]
            #graph_repr = F.relu(self.graph_pooling(graph_repr))
            prediction = self.predictor(graph_repr)  # [1]

            batch_outputs.append(prediction)

        # Stack individual predictions - inefficient compared to true batching
        return torch.stack(batch_outputs)

## 1.3 Model training
Now we can train the model using the standard SGD approach. We'll use CrossEntropyLoss as we are performing multiclass classification .

### Note: This will likely take a while to run - more on this to come.

In [ ]:
# data loaders
batch_size = 4

# create model instance
seed_everything(RANDOM_STATE)
model = GraphSequenceTransformer(node_feature_dim=dm.num_node_features, pe_dim=pe_dim, num_classes=dm.num_classes, hidden_dim=64, num_heads=4).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = tnn.CrossEntropyLoss()

# training loop
num_epochs = 10
for epoch in range(num_epochs):  # Just a few epochs for demo
        model.train()
        total_loss = 0

        for batch_idx, batch in enumerate(dm.train_dataloader()):
            optimizer.zero_grad()
            batch = batch.to(device)

            # Forward pass - processes each graph individually!
            predictions = model(batch)
            loss = criterion(predictions.squeeze(), batch.y)

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in dm.val_dataloader():
                predictions = model(batch.to(device))
                val_loss += criterion(predictions.squeeze(), batch.y).item()

        avg_train_loss = total_loss / len(dm.train_dataloader())
        avg_val_loss = val_loss / len(dm.val_dataloader())
        print(f'Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}')



## 1.4 Model Evaluation

Although we haven't done any parameter tuning, for purpose of illustration, let's examine the model performance on the test set.

In [ ]:
# evaluate on test set
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for batch in dm.test_dataloader():
        batch = batch.to(device)
        predictions = model(batch)
        all_preds.append(predictions.cpu())
        all_labels.append(batch.y.cpu())
all_preds = torch.cat(all_preds, dim=0)
all_labels = torch.cat(all_labels, dim=0)
predicted_classes = all_preds.argmax(dim=1)
class_names = [enz_class_map[i] for i in range(dm.num_classes)]
print("\nClassification Report:")
print(classification_report(all_labels, predicted_classes, target_names=class_names))

## Why is this a "bad" approach?
Consider the `forward` method in our `GraphSequenceTransformer`. In that method, we iterate over the graphs in the batch and process them one at a time. We are not processing the minibatch as a matrix, but rather as one sample graph at a time. This presents significant computational inefficiency. We append the predictions to a list before returning them to the training loop, so we are still performing minibatch gradient descent. Can we perhaps overcome processing one graph at a time? How should we represent a batch for the Transformer? The block diagonal adjacency matrix used in PyG does not work becuase the Transformer does not use an adjacency matrix in its computations. We could instead assume a `max_node` size and pad shorter graphs with `zero` nodes such as the following:

```
def forward(self, batch):
        graphs = batch.to_data_list()
        batch_tensor = torch.zeros(len(graphs), self.max_nodes, self.node_embedding.out_features)
        attention_mask = torch.ones(len(graphs), self.max_nodes, dtype=torch.bool)
        
        for i, graph in enumerate(graphs):
            num_nodes = graph.x.size(0)
            if num_nodes > self.max_nodes:
                # Truncate if too large
                num_nodes = self.max_nodes
                
            x = torch.cat([graph.x[:num_nodes], graph.pe[:num_nodes]], dim=-1)
            x = self.node_embedding(x)
            
            batch_tensor[i, :num_nodes] = x
            attention_mask[i, :num_nodes] = False  # False means "don't mask"
            
        # Now we can use standard transformer with proper batching
        output = self.transformer(batch_tensor, src_key_padding_mask=attention_mask)
        return output
```

 Notice how every the feature vectors and position encodings for every graph is assigned to a row in the `batch_tensor` that is `[max_nodes, embedding]`. For many graphs the entries will correspond to non-existent _ghost_ nodes. These create a significant issues:
 * diluted attention scores: When a 10-node graph is padded to 100, each real node's attention gets distributed across 90 meaningless padding positions
 * over-fitting: The model might learn spurious correlations with padding rather than meaningful graph patterns

 Finally, while the positional encodings help with position aware tasks, we would like to include adjacency information.

 ### Let's see how we can do better.

---
# Part 2: Node Classification with Hybrid Graph Transformers
To overcome the issues identified in Part 1, we can consider hybrid Graph Transformer layers that combine elements of message passing with elements of Transformers. We will specifically consider the [TransformerConv](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.nn.conv.TransformerConv.html#torch_geometric.nn.conv.TransformerConv) layer in PyG. This GNN is similar to `GATConv` in that it uses message passing so that a given node only attends to its neighbor nodes and that in message aggregation it applies learnable weights to each neighbor. It differs in that it constructs the learnable weights using the full query, key, and value matrix approach developed for Transformers.

For more information, see [Masked Label Prediction: Unified Message Passing Model for Semi-Supervised Classification](https://arxiv.org/pdf/2009.03509)

## 2.1 Load the PubMed Dataset
We will first apply this approach to node classification on PubMed articles. The PubMed dataset is a citation network with nearly 20K articles related to diabetes. Each article is in one of three classes 'Diabetes Mellitus Experimental', 'Diabetes Mellitus 1', 'Diabetes Mellitus 2'.

In [ ]:
dataset = Planetoid(root=data_dir, name='PubMed', split='public')
data = dataset[0]

# size of training set
print("Training set size:", data.train_mask.sum().item())
print("Validation set size:", data.val_mask.sum().item())
print("Test set size:", data.test_mask.sum().item())

print(f"Number of classes: {dataset.num_classes}")
print(f"Number of node features: {dataset.num_node_features}")
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of edges: {data.num_edges}")

## 2.2 Positional encodings with Laplacian eigenvectors
In our hybrid GNN + Transformer model, we can optionally use postional encoding. Combining the adjacency information via local attention with global position information in the positional encoding can enhance model performance.

We will add Graph Laplacian PEs to the input node features. Conveniently, the PyG [AddLaplacianEigenvectorsPE](https://pytorch-geometric.readthedocs.io/en/2.5.2/generated/torch_geometric.transforms.AddLaplacianEigenvectorPE.html#torch_geometric.transforms.AddLaplacianEigenvectorPE) transform will handle this for us. The key input features for the transform are:
*   `k` - the number of non-unit eigenvectors to consider
*   `attr_name` - the key for the PEs in `data`

In [ ]:
test_k = 8
glpe_transform = AddLaplacianEigenvectorPE(k=test_k, attr_name='pe')
data = glpe_transform(data)

# lets also split the data with RandomNodeSplit
N_test = int(data.num_nodes * 0.2)

N_val = int(data.num_nodes * 0.1)
rns_transform = RandomNodeSplit(num_val=N_val, num_test=N_test, split='train_rest')
data = rns_transform(data)
print(data)

## 2.3 Building the Hybrid GNN Transformer encoder and decoder

Here we will build our graph transformer encoder. The output of the encoder will be a node embedding matrix as we've seen in our previous labs. We will apply the PyG [TransformerConv](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.nn.conv.TransformerConv.html#torch_geometric.nn.conv.TransformerConv) transformer described in the [“Masked Label Prediction: Unified Message Passing Model for Semi-Supervised Classification”](https://arxiv.org/abs/2009.03509) paper. We will choose two such layers, which implies that the receptive field of each node is its two hop neighborhood.

**Note** that we need to set the input dimension of the `TransformerConv` to be the original node feature dimension from the PubMed dataset plus the size of the positional encoding vector as we are concatenating those to form a single feature vector for the nodes. Notice also, that the `forward` method we pass in both the feature vector `x` and the positional encoding `pe` which we concatenate before passing to the TransformerConv.

In [ ]:
class GraphTransformerEncoder(tnn.Module):
    def __init__(self, feature_dim, pe_dim, embedding_dim, num_heads=2, dropout=0.2):
        super(GraphTransformerEncoder, self).__init__()
        indim = feature_dim + pe_dim
        self.indim = indim

        self.conv1 = TransformerConv(in_channels=self.indim, out_channels=embedding_dim, heads=num_heads, dropout=dropout, concat=False)
        self.conv2 = TransformerConv(in_channels=embedding_dim, out_channels=embedding_dim, heads=num_heads, concat=False, dropout=dropout)

    def forward(self, x, pe, edge_index):
        if pe is not None:
            x = torch.cat([x, pe], dim=-1)
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        return x

# test the model dimensions
model = GraphTransformerEncoder(feature_dim=dataset.num_node_features, pe_dim=test_k, embedding_dim=64)
out = model(data.x, data.pe, data.edge_index)
print(out.shape)  # should be (num_nodes, embedding_dim)

As we are performing classification, we will project the final node embeddings from the encoder through a linear layers. In the class below, we pass in an encoder (which is expected to produce the node embeddings) and pass its output through the linear layer. We will use cross entropy loss, noting that the PyTorch implementation will handle the softmax computation.

In [ ]:
class GraphTransformerClassifier(tnn.Module):
    def __init__(self, encoder, embedding_dim, num_classes):
        super(GraphTransformerClassifier, self).__init__()
        self.encoder = encoder
        self.lin = tnn.Linear(embedding_dim, num_classes)

    def forward(self, x, pe, edge_index):
        x = self.encoder(x, pe, edge_index)
        x = self.lin(x)
        return x

# create a test sample
embedding_dim = 64
encoder = GraphTransformerEncoder(feature_dim=dataset.num_node_features, pe_dim=data.pe.shape[1], embedding_dim=64)
model = GraphTransformerClassifier(encoder=encoder, embedding_dim=embedding_dim, num_classes=dataset.num_classes)
out = model(data.x, data.pe, data.edge_index)
print(out.shape)  # should be (num_nodes, num_classes)

## 2.4 Training the model
Now we can train our model. We'll follow our standard training procedure. The key difference of this training method relative to previous labs is that we need to pass the positional encoding into the call to `model`.

In [ ]:
def train(model, data, max_epochs=50):
    # Prepare data
    seed_everything(RANDOM_STATE, verbose=False)
    x = data.x.to(device)
    if hasattr(data, 'pe') and data.pe is not None:
        pe = data.pe.to(device)
    else:
        pe = None
    edge_index = data.edge_index.to(device)
    y = data.y.to(device)

    # Define model, loss, optimizer
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad()
        out = model(x, pe, edge_index)
        loss = F.cross_entropy(out[data.train_mask], y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            pred = out.argmax(dim=1)
            train_acc = (pred[data.train_mask] == y[data.train_mask]).sum().item() / data.train_mask.sum().item()
            val_acc = (pred[data.val_mask] == y[data.val_mask]).sum().item() / data.val_mask.sum().item()

        if epoch % 10 == 0 or epoch == max_epochs - 1:
            print(f'Epoch {epoch:03d}, Loss: {loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}')

    return model, val_acc

# Let's test our training function
embedding_dim = 64
encoder = GraphTransformerEncoder(feature_dim=dataset.num_node_features, pe_dim=test_k, embedding_dim=embedding_dim, num_heads=4, dropout=0.2)
model = GraphTransformerClassifier(encoder=encoder, embedding_dim=embedding_dim, num_classes=dataset.num_classes)
model, val_acc = train(model, data, max_epochs=20)

## 2.5 Hyperparameter tuning
We can consider many different hyperparameters for tuning. Here, let's consider the number of eigenvectors used in the positional encoding to see how that changes our model performance.

In [ ]:
K = [None, 2, 4, 8, 16]
best_k = None
best_val_acc = 0
best_model = None
max_epochs = 50 # more epochs may lead to better performance

# create the random node split transform outside the loop to ensure same splits for each k
N_test = int(data.num_nodes * 0.2)
N_val = int(data.num_nodes * 0.1)
seed_everything(RANDOM_STATE)
rns_transform = RandomNodeSplit(num_val=N_val, num_test=N_test, split='train_rest')
for k in K:
    dataset = Planetoid(root=data_dir, name='PubMed', split='public')
    data = dataset[0]
    # create positional encodings using k eigenvectors if k is not None
    if k is not None:
        glpe_transform = AddLaplacianEigenvectorPE(k=k, attr_name='pe')
        data = glpe_transform(data)
        pe_dim = k
    else:
        pe_dim = 0
    # apply random node split
    data = rns_transform(data)

    embedding_dim = 64
    encoder = GraphTransformerEncoder(feature_dim=dataset.num_node_features, pe_dim=pe_dim, embedding_dim=embedding_dim, num_heads=4, dropout=0.2)
    model = GraphTransformerClassifier(encoder=encoder, embedding_dim=embedding_dim, num_classes=dataset.num_classes)
    print(f"\nTraining with k={k} eigenvectors for positional encoding")
    model, val_acc = train(model, data, max_epochs=max_epochs)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_k = k
        best_model = model
print(f"Best k: {best_k} with Validation Accuracy: {best_val_acc:.4f}")

## 2.6 Model evaluation
Finally, we can evaluate the model on the test.

In [ ]:
def test(model, data):
    # Prepare data
    x = data.x.to(device)
    pe = data.pe.to(device)
    edge_index = data.edge_index.to(device)
    y = data.y.to(device)

    # get sample predictions
    model.eval()
    with torch.no_grad():
        proba = model(x, pe, edge_index)
        y_true = y[data.test_mask]
        y_proba = proba[data.test_mask]
        y_pred = F.softmax(proba, dim=-1)[data.test_mask].argmax(dim=1)

    return y_true, y_proba, y_pred


dataset = Planetoid(root=data_dir, name='PubMed', split='public')
data = dataset[0]
# create positional encodings using k eigenvectors
glpe_transform = AddLaplacianEigenvectorPE(k=best_k, attr_name='pe')
data = glpe_transform(data)

# apply the same random node split transform we used during training to ensure we have the proper test set
data = rns_transform(data)

y_true, y_proba, y_pred = test(best_model, data)

# convert to numpy
y_true_np = y_true.cpu().numpy()
y_proba_np = y_proba.cpu().numpy()
y_pred_np = y_pred.cpu().numpy()

# print the classification report
class_dict = {0: 'Diabetes Mellitus Exp', 1: 'Diabetes Mellitus 1', 2: 'Diabetes Mellitus 2'}
class_names = [v for k, v in class_dict.items()]
print(classification_report(y_true_np, y_pred_np, target_names = class_names))

# ROC curves for each class - functions now handle device conversion automatically
plot_roc_curves(y_true_np, y_proba_np, class_names,
                title="ROC Curves By Class")

# Confusion matrix - functions now handle device conversion automatically
plot_confusion_matrix(y_true_np, y_pred_np, class_names,
                     title="Confusion Matrix")

In [ ]:
# clean up
del dataset, data, model, encoder, best_model

---
# Part 3 Graph regression with hybrid GNN Transformers

Here, we will use the same encoder model we created above for node classifcation. We will create a new prediction head for regression. We will examine peformance on the ZINC dataset from the ZINC database and the “Automatic Chemical Design Using a Data-Driven Continuous Representation of Molecules” paper, containing about 250,000 molecular graphs with up to 38 heavy atoms. The task is to regress the penalized logP (also called constrained solubility in some works), given by y = logP - SAS - cycles, where logP is the water-octanol partition coefficient, SAS is the synthetic accessibility score, and cycles denotes the number of cycles with more than six atoms.


## 3.1 The ZINC dataset
First, let's examine the ZINC dataset. We will use the subset option which contains 12K graphs, 10K for training and 1K each for validation and test.

In [ ]:
seed_everything(RANDOM_STATE)
walk_length = 20
rwtransform = AddRandomWalkPE(walk_length=walk_length, attr_name='pe')
train_dataset = ZINC(root='/tmp/ZINC', subset=True, split='train', transform=rwtransform)
val_dataset = ZINC(root='/tmp/ZINC', subset=True, split='val', transform=rwtransform)
test_dataset = ZINC(root='/tmp/ZINC', subset=True, split='test', transform=rwtransform)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
cnt = 0
max_cnt = 2
for step, data in enumerate(train_loader):
    print(f'Step {step + 1}:')
    print('=======')
    print(f'Number of graphs in the current batch: {data.num_graphs}')
    print(data)
    print()
    cnt += 1
    if cnt == max_cnt:
        break

## 3.2 Build the GNN Transformer Regressor

In [ ]:
class GraphTransformerRegressor(tnn.Module):
    def __init__(self, encoder, embedding_dim):
        super(GraphTransformerRegressor, self).__init__()
        self.encoder = encoder
        self.lin = tnn.Linear(embedding_dim, 1)

    def forward(self, x, pe, edge_index, batch):
        x = self.encoder(x, pe, edge_index)
        x = global_mean_pool(x, batch)  # Global mean pooling
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.lin(x)
        return x

In [ ]:
seed_everything(RANDOM_STATE)
encoder = GraphTransformerEncoder(feature_dim=train_dataset.num_node_features, pe_dim=walk_length, embedding_dim=64, num_heads=4, dropout=0.2)
model = GraphTransformerRegressor(encoder=encoder, embedding_dim=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.MSELoss()

Now let's train the model.

In [ ]:
num_epochs = 5
for epoch in range(num_epochs):  # Just a few epochs for demo
        model.train()
        total_loss = 0

        for batch_idx, batch in enumerate(train_loader):
            optimizer.zero_grad()
            x = batch.x.to(device)
            pe = batch.pe.to(device)
            edge_index = batch.edge_index.to(device)
            b = batch.batch.to(device)
            predictions = model(x, pe, edge_index, b)
            y = batch.y.to(device)
            loss = criterion(predictions.squeeze(), y)

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

            if batch_idx % 500 == 0:
                print(f'Epoch {epoch+1}, Batch {batch_idx}, Loss: {loss.item():.4f}')

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                x = batch.x.to(device)
                pe = batch.pe.to(device)
                edge_index = batch.edge_index.to(device)
                b = batch.batch.to(device)
                predictions = model(x, pe, edge_index, b)
                y = batch.y.to(device)
                val_loss += criterion(predictions.squeeze(), y).item()

        avg_train_loss = total_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        print(f'Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}\n')

Let's evaluate the test set performance

In [ ]:
def test(model, data_loader):
    model.eval()
    test_mse = 0
    residuals = []
    y_true = np.zeros(len(test_loader.dataset))
    residuals = np.zeros(len(test_loader.dataset))
    idx = 0
    with torch.no_grad():
        for batch in data_loader:
            x = batch.x.to(device)
            pe = batch.pe.to(device)
            edge_index = batch.edge_index.to(device)
            b = batch.batch.to(device)
            predictions = model(x, pe, edge_index, b)
            y = batch.y.to(device)
            test_mse += criterion(predictions.squeeze(), y).item()
            residuals[idx:idx + len(y)] = (predictions.squeeze() - y).cpu().numpy()
            y_true[idx:idx + len(y)] = y.cpu().numpy()
            idx += len(y)

    avg_test_mse = test_mse / len(val_loader)
    print(f'Test Average MSE: {avg_test_mse:.4f}\n')

    # plot the residual error distribution
    # residuals = torch.cat(residuals, dim=0).numpy()
    plt.figure(figsize=(8,6))
    sns.histplot(residuals, bins=50, kde=True)
    plt.title('Residual Errors Distribution')
    plt.xlabel('Residual Error')
    plt.ylabel('Frequency')
    plt.grid(True, alpha=0.3)
    plt.show()


    print(len(y_true), len(residuals))
    # y_true = torch.cat(y_true, dim=0).numpy()
    # scatter plot of true vs residuals
    plt.figure(figsize=(8,6))
    plt.scatter(y_true, residuals, alpha=0.5)
    plt.axhline(0, color='red', linestyle='--')
    plt.title('Residuals Scatter Plot')
    plt.xlabel('y_true')
    plt.ylabel('Residual Error')
    plt.grid(True, alpha=0.3)
    plt.show()

test(model, test_loader)